In [1]:
!pip install mlflow
!pip install xgboost
!pip install lifetimes

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Import 
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from mlflow.tracking import MlflowClient
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

In [3]:
#Load the data 
df = pd.read_parquet('D:\\Projects\\Customer_Retention\\data\\02_rfm_features.parquet', engine='pyarrow')
df.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Segment,RFM_Score,Segment_Label
0,12346,326,12,77556.46,2,5,5,255,12,Loyal Customers
1,12347,2,8,4921.53,5,4,5,545,14,Champions
2,12348,75,5,2019.40,3,4,4,344,11,Loyal Customers
3,12349,19,4,4428.69,5,3,5,535,13,Champions
4,12350,310,1,334.40,2,1,2,212,5,At Risk


In [4]:
#defining churn based on recency 
CHURN_THRESHOLD = 90

df['Churned'] = (df['Recency'] > CHURN_THRESHOLD).astype(int)

In [5]:
churn_counts = df['Churned'].value_counts()
churn_percentages = df["Churned"].value_counts(normalize=True) * 100

In [6]:
print("Count of customers:\n", churn_counts, "\n")
print("Percentage of customers:\n", churn_percentages.round(2))

Count of customers:
 Churned
1    2989
0    2889
Name: count, dtype: int64 

Percentage of customers:
 Churned
1    50.85
0    49.15
Name: proportion, dtype: float64


In [7]:
# Spliting the datasets for model training and testing the model

features = ['Frequency', 'Monetary', 'F_Score', 'M_Score']
X = df[features]
y = df['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
print(f"Total customers: {len(df)}")
print(f"Training set size: {len(X_train)} customers")
print(f"Testing set size: {len(X_test)} customers")

Total customers: 5878
Training set size: 4702 customers
Testing set size: 1176 customers


In [9]:
db_path = r"D:\\Projects\\Customer_Retention\\mlflow.db"
tracking_uri = f"sqlite:///{db_path.replace('\\', '/')}"
mlflow.set_tracking_uri(tracking_uri)

In [10]:
experiment_name = "Customer_Churn_Prediction"
mlflow.set_experiment(experiment_name)
print("MLflow is successfully connected!")
print(f"Database located at: {mlflow.get_tracking_uri()}")

MLflow is successfully connected!
Database located at: sqlite:///D://Projects//Customer_Retention//mlflow.db


In [11]:
with mlflow.start_run(run_name="Logistic_Regression_Baseline"):
    

    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(X_train, y_train)

    y_pred = lr_model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("model_type", "Logistic Regression")

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(lr_model, "model")
    
    print("Model trained and logged to MLflow!")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

2026/03/27 15:48:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 15:48:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model trained and logged to MLflow!
Accuracy:  0.6811
Precision: 0.6554
Recall:    0.7615
F1 Score:  0.7045


In [12]:
with mlflow.start_run(run_name="Random_Forest"):
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_model.fit(X_train, y_train)
    
    y_pred_rf = rf_model.predict(X_test)
    f1_rf = f1_score(y_test, y_pred_rf)
    
    mlflow.log_params({"n_estimators": 100, "max_depth": 5, "model_type": "Random Forest"})
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "precision": precision_score(y_test, y_pred_rf),
        "recall": recall_score(y_test, y_pred_rf),
        "f1_score": f1_rf
    })
    mlflow.sklearn.log_model(rf_model, "model")
    print(f"Random Forest trained! F1 Score: {f1_rf:.4f}")

2026/03/27 15:48:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 15:48:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest trained! F1 Score: 0.7118


In [13]:
with mlflow.start_run(run_name="XGBoost"):
    xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
    xgb_model.fit(X_train, y_train)
    
    y_pred_xgb = xgb_model.predict(X_test)
    f1_xgb = f1_score(y_test, y_pred_xgb)
    
    mlflow.log_params({"n_estimators": 100, "learning_rate": 0.1, "max_depth": 5, "model_type": "XGBoost"})
    mlflow.log_metrics({
        "accuracy": accuracy_score(y_test, y_pred_xgb),
        "precision": precision_score(y_test, y_pred_xgb),
        "recall": recall_score(y_test, y_pred_xgb),
        "f1_score": f1_xgb
    })
    mlflow.xgboost.log_model(xgb_model, "model")
    print(f"XGBoost trained! F1 Score: {f1_xgb:.4f}")

2026/03/27 15:48:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost trained! F1 Score: 0.7097


In [14]:
PROJECT_ROOT = Path("D:/Projects/Customer_Retention")
LOCAL_MLRUNS = PROJECT_ROOT / "mlruns"
LOCAL_MLRUNS.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT}/mlflow.db")

experiment = mlflow.get_experiment_by_name("Customer_Churn_Prediction")
best_run = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_score DESC"],
    max_results=1
).iloc[0]

best_run_id = best_run.run_id
best_model_type = best_run["params.model_type"]

print(f"Best model: {best_model_type} | Run ID: {best_run_id}")

model_uri = f"runs:/{best_run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

mlflow.set_experiment("Customer_Churn_Prediction")

with mlflow.start_run(run_name="Best_Model_Local_Export") as run:
    mlflow.set_tag("exported_from_run", best_run_id)
    mlflow.set_tag("model_type", best_model_type)

    mlflow.sklearn.log_model(
        sk_model=loaded_model,
        artifact_path="model",
        registered_model_name="Best_Churn_Predictor",
    )

    export_run_id = run.info.run_id
    artifact_uri = run.info.artifact_uri

print(f"\nModel saved locally!")
print(f"   Run ID       : {export_run_id}")
print(f"   Artifact URI : {artifact_uri}")
print(f"   mlruns path  : {LOCAL_MLRUNS}")

mlmodel_files = list(LOCAL_MLRUNS.rglob("MLmodel"))
print(f"   MLmodel files found: {len(mlmodel_files)}")
for f in mlmodel_files:
    print(f"   → {f}")

Best model: Random Forest | Run ID: 3abf9a4fda11459a91a5d7d31f09b0b2


2026/03/27 15:49:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/27 15:49:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Model saved locally!
   Run ID       : 53246f7335334b3f94282d76a2ce5025
   Artifact URI : file:C:/Users/viraj/mlruns/1/53246f7335334b3f94282d76a2ce5025/artifacts
   mlruns path  : D:\Projects\Customer_Retention\mlruns
   MLmodel files found: 0


Registered model 'Best_Churn_Predictor' already exists. Creating a new version of this model...
Created version '4' of model 'Best_Churn_Predictor'.


In [15]:
experiment = mlflow.get_experiment_by_name("Customer_Churn_Prediction")
best_run = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1_score DESC"],
    max_results=1
).iloc[0]

best_run_id = best_run.run_id
best_f1 = best_run["metrics.f1_score"]
best_name = best_run["tags.mlflow.runName"]

print(f"Best Model Found: {best_name} (F1 Score: {best_f1:.4f})")

model_uri = f"runs:/{best_run_id}/model"
mlflow.register_model(model_uri=model_uri, name="Best_Churn_Predictor")

print("Best model successfully registered!")

Best Model Found: Random_Forest (F1 Score: 0.7118)


Registered model 'Best_Churn_Predictor' already exists. Creating a new version of this model...
2026/03/27 15:49:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 3abf9a4fda11459a91a5d7d31f09b0b2 has no artifacts at artifact path 'model', registering model based on models:/m-e9014a3a444f47a08b195659bf3e906a instead


Best model successfully registered!


Created version '5' of model 'Best_Churn_Predictor'.


In [16]:
df_raw = pd.read_parquet(r'D:\\Projects\\Customer_Retention\data\\01_cleaned_retail.parquet')

clv_summary = summary_data_from_transaction_data(
    df_raw, 
    customer_id_col='Customer ID', 
    datetime_col='InvoiceDate', 
    monetary_value_col='Revenue',
    observation_period_end=df_raw['InvoiceDate'].max()
)

clv_summary = clv_summary[clv_summary['frequency'] > 0]
clv_summary.head()

,frequency,recency,T,monetary_value
Customer ID,,,,
12346,7.0,400.0,725.0,11066.637143
12347,7.0,402.0,404.0,615.714286
12348,4.0,363.0,438.0,449.310000
12349,3.0,571.0,589.0,1120.056667
12352,8.0,356.0,392.0,338.261250


In [17]:
with mlflow.start_run(run_name="CLV_Estimation_Lifetimes"):

    bgf = BetaGeoFitter(penalizer_coef=0.01)
    bgf.fit(clv_summary['frequency'], clv_summary['recency'], clv_summary['T'])

    ggf = GammaGammaFitter(penalizer_coef=0.01)
    ggf.fit(clv_summary['frequency'], clv_summary['monetary_value'])

    clv_summary['predicted_clv'] = ggf.customer_lifetime_value(
        bgf,
        clv_summary['frequency'],
        clv_summary['recency'],
        clv_summary['T'],
        clv_summary['monetary_value'],
        time=6, 
        discount_rate=0.01
    )

    avg_clv = clv_summary['predicted_clv'].mean()
    mlflow.log_metric("average_6m_clv", avg_clv)
    mlflow.log_param("clv_horizon_months", 6)
    
    print(f"CLV Estimation Complete. Average Predicted 6-Month CLV: ${avg_clv:.2f}")

CLV Estimation Complete. Average Predicted 6-Month CLV: $1071.40
